In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import IPython.display as ipd
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import GridSearchCV, cross_val_score
from glob import glob
from itertools import cycle
import seaborn as sns
from pathlib import Path
from collections import Counter
import joblib

sns.set_theme(style="white", palette=None)

In [ ]:
def cleanUp(audioInfo):
   X = [item["features"] for item in audioInfo]
   Y = [item["chord"] for item in audioInfo]
   X = np.array(X)
   Y = np.array(Y)
   # print(X.shape)
   # print(Y.shape)
   # print(Counter(Y))
   return X, Y


In [ ]:
def extract_features(y, sr):
    target_duration = 5
    if len(y) > target_duration * sr:
        y_fixed = y[: target_duration * sr]
    else:
        padding = (target_duration * sr) - len(y)
        y_fixed = np.pad(y, (0, padding), mode="constant")

    y_harmonic = librosa.effects.harmonic(y_fixed)

    # Skip the sharpest ~80ms of pick-attack transient noise before measuring
    # chroma — this reduces spurious energy bleeding into unrelated pitch
    # classes from the initial strike, letting the strings' actual sustained
    # pitch content dominate the measurement instead.
    skip_samples = int(sr * 0.08)
    y_for_chroma = y_harmonic[skip_samples:] if len(y_harmonic) > skip_samples else y_harmonic

    chroma = librosa.feature.chroma_cqt(y=y_for_chroma, sr=sr)
    tonnetz = librosa.feature.tonnetz(y=y_for_chroma, sr=sr)

    return np.concatenate([
        chroma.mean(axis=1), chroma.std(axis=1),
        tonnetz.mean(axis=1), tonnetz.std(axis=1),
    ])

In [ ]:
# def create_dataset(audioFiles):
#     audioInfo = []
#     target_duration = 5
#     for file in audioFiles:
#         y, sr = librosa.load(file)
#         if(len(y) > target_duration * sr):
#             y_fixed = y[:target_duration * sr]
#         else:
#             padding = (target_duration * sr) - len(y)
#             y_fixed = np.pad(y, (0, padding), mode="constant")
#         S = librosa.feature.melspectrogram(y=y_fixed, sr=sr, n_mels=128)
#         S_db_mel = librosa.amplitude_to_db(S, ref=np.max)
#         chordName = Path(file).parent.name
#         audioInfo.append({"features": S_db_mel, "chord": chordName})
#     return cleanUp(audioInfo)

In [ ]:
def create_dataset(audioFiles):
    audioInfo = []
    for file in audioFiles:
        y, sr = librosa.load(file)
        features = extract_features(y, sr)
        chordName = Path(file).parent.name
        audioInfo.append({"features": features, "chord": chordName})
    return cleanUp(audioInfo)

In [ ]:
trainFiles = glob("./raw_dataset/Train/*/*.wav")
testFiles = glob("./raw_dataset/Test/*/*.wav")
X_train, Y_train = create_dataset(trainFiles)
X_test, Y_test = create_dataset(testFiles)

print("Training files found:", len(trainFiles))
print("Test files found:", len(testFiles))
print(Counter(Y_train))

In [ ]:
encoder = LabelEncoder()
y_train_encoded = encoder.fit_transform(Y_train)
y_test_encoded = encoder.transform(Y_test)

print("Classes:", encoder.classes_)
print("Feature vector length:", X_train.shape[1])

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", probability=True)),
])

param_grid = {
    "svm__C": [1, 10, 50, 100],
    "svm__gamma": ["scale", 0.01, 0.001],
}

grid = GridSearchCV(pipeline, param_grid, cv=5)
grid.fit(X_train, y_train_encoded)

model = grid.best_estimator_
print("Best params:", grid.best_params_)

accuracy = model.score(X_test, y_test_encoded)
print(f"Test accuracy: {accuracy:.2f}")

scores = cross_val_score(
    model,
    np.vstack([X_train, X_test]),
    np.concatenate([y_train_encoded, y_test_encoded]),
    cv=5,
)
print(f"Cross-val accuracy: {scores.mean():.2f} ± {scores.std():.2f}")

In [ ]:
# joblib.dump(model, "chord_model.pkl")
# joblib.dump(encoder, "chord_encoder.pkl")

In [ ]:
def testAudio(file):
    y, sr = librosa.load(file)
    features = extract_features(y, sr).reshape(1, -1)

    probs = model.predict_proba(features)[0]
    sorted_idx = np.argsort(probs)[::-1]
    best_idx, second_idx = sorted_idx[0], sorted_idx[1]

    chord = encoder.classes_[best_idx]
    second_chord = encoder.classes_[second_idx]
    confidence = probs[best_idx]
    margin = confidence - probs[second_idx]

    return f"{chord} ({confidence:.1%}) vs {second_chord} ({probs[second_idx]:.1%}), margin {margin:.1%}"

In [ ]:
myPlays = glob("./MyPlays/*.wav")
for f in myPlays:
    print(f"{Path(f).name}: {testAudio(f)}")